In [14]:
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

In [15]:
transform=transforms.ToTensor() # convert image into tensor 
train_data=datasets.MNIST(
    root='data',
    train=True,
    download=True,
    transform=transform
)

test_data=datasets.MNIST(
    root='data',
    train=False,
    download=True,
    transform=transform
)

In [16]:
train_loader=DataLoader(train_data,batch_size=32)
test_loader=DataLoader(test_data,batch_size=32)

## CNN

In [17]:
import torch.nn as  nn
import torch.optim as optim

In [30]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layer=nn.Sequential(
            # frist layer
            nn.Conv2d(in_channels=1,out_channels=32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
             # second layer
            nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc_layers=nn.Sequential(
            nn.Linear(64*7*7,256),
            nn.ReLU(),
            nn.Linear(256,10) 
        )
    def forward(self,x):
            x=self.conv_layer(x)
            x=x.view(x.size(0),-1)
            x=self.fc_layers(x)
            return x

In [31]:
model=CNN()
critenion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

#### Traninig the CNN

In [32]:
epochs=10
for epoch in range(epochs):
    epoch_training_loss=0.0
    for image ,labels in train_loader:
        optimizer.zero_grad()
        output=model.forward(image)
        loss=critenion(output,labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss+=loss.item()
    print(f"epoch={epoch+1} and loss={epoch_training_loss/len(train_loader)}")
    

epoch=1 and loss=0.14521190136709095
epoch=2 and loss=0.04318306268239782
epoch=3 and loss=0.026476570317553585
epoch=4 and loss=0.01812951513173445
epoch=5 and loss=0.014236192196279595
epoch=6 and loss=0.010315765620331069
epoch=7 and loss=0.007707043918332492
epoch=8 and loss=0.007340482078829331
epoch=9 and loss=0.007611541983846319
epoch=10 and loss=0.006132371974314479


### Evaluation of Model

In [33]:
import torch

correct_labels = 0
total_labels = 0

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)   
        
        _, predicted = torch.max(outputs, 1)
        
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)   
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
accuracy = (correct_labels / total_labels) * 100
print(f"Accuracy = {accuracy:.2f}%")

Accuracy = 99.07%


In [34]:
from sklearn.metrics import confusion_matrix,classification_report

In [35]:
cm = confusion_matrix(all_labels, all_preds)
print(cm)

[[ 979    0    0    0    0    0    0    1    0    0]
 [   0 1132    1    0    1    0    1    0    0    0]
 [   1    3 1019    2    1    0    1    4    1    0]
 [   0    1    0 1004    0    3    0    1    1    0]
 [   0    0    0    0  971    0    5    0    2    4]
 [   1    0    0    3    0  887    1    0    0    0]
 [   7    2    0    0    0    1  947    0    1    0]
 [   0    5    5    1    2    0    0 1013    0    2]
 [   4    0    1    0    0    2    0    0  966    1]
 [   1    0    0    0   10    4    0    4    1  989]]


In [36]:
cm=classification_report(all_labels,all_preds)
print(cm)

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       980
           1       0.99      1.00      0.99      1135
           2       0.99      0.99      0.99      1032
           3       0.99      0.99      0.99      1010
           4       0.99      0.99      0.99       982
           5       0.99      0.99      0.99       892
           6       0.99      0.99      0.99       958
           7       0.99      0.99      0.99      1028
           8       0.99      0.99      0.99       974
           9       0.99      0.98      0.99      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



## RNN

In [99]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layer=1):
        super().__init__()
        self.hidden_size=hidden_size
        self.num_layer=num_layer
        #Rnn Layer
        self.rnn=nn.RNN(input_size,hidden_size,num_layer,batch_first=True)

        #fully connected layer
        self.fc=nn.Linear(hidden_size,10)

    def forward(self,x):
        h0=torch.zeros(self.num_layer,x.size(0),self.hidden_size)
        out,_=self.rnn(x,h0)
        out=self.fc(out[:,-1,:])
        return out
    

In [100]:
image,label=train_data[0]
input_size=image.shape[1]

In [101]:
model=RNN(input_size)
critersion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

### Training RNN 

In [102]:
epochs=10
for epoch in range(epochs):
    model.train()
    for xb,yb in train_loader:
        optimizer.zero_grad()
        xb=xb.squeeze(1)
        
        output=model(xb).squeeze()
        output=torch.sigmoid(output)
        loss=critersion(output,yb)
        loss.backward()
        optimizer.step()
    print(f"{epoch+1}/{epochs} amd loss={loss.item()}")

torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size([32, 28, 28])
torch.Size([32, 10])
torch.Size([32, 1, 28, 28])
torch.Size

KeyboardInterrupt: 